# Project Atlas — COLMAP + gsplat (Colab T4)

**What this notebook does:**
1. Upload `frames_for_colab.zip` (extracted by `capture.py` locally)
2. Run COLMAP SfM (camera poses + sparse point cloud)
3. Train Gaussian Splat via nerfstudio
4. Export `.splat` file → download as `splat_result.zip`

**After download:** unzip into `data/scans/{scan_id}/splat/` locally

**Expected time:** ~15 min total on free T4 for a single room (30–60 frames)

---
**Before running:** Runtime → Change runtime type → T4 GPU



In [5]:
# ==========================================================
# Cell 1 — Verify Google Colab GPU
# ==========================================================

import platform
import subprocess
import torch

print("=" * 80)
print("System Information")
print("=" * 80)

print(f"OS             : {platform.platform()}")
print(f"Python Version : {platform.python_version()}")
print(f"PyTorch Version: {torch.__version__}")

print("\nChecking CUDA...")

if not torch.cuda.is_available():
    raise RuntimeError(
        "❌ CUDA GPU not detected.\n"
        "Go to Runtime → Change runtime type → GPU (Tesla T4)"
    )

gpu = torch.cuda.get_device_properties(0)

print(f"✅ GPU Name      : {gpu.name}")
print(f"✅ Compute Cap. : {gpu.major}.{gpu.minor}")
print(f"✅ VRAM         : {gpu.total_memory/1024**3:.2f} GB")

print("\nNVIDIA Driver")
print("-"*80)

subprocess.run(["nvidia-smi"])

print("\n✅ Environment Ready")

System Information
OS             : Linux-6.6.122+-x86_64-with-glibc2.35
Python Version : 3.12.13
PyTorch Version: 2.11.0+cu128

Checking CUDA...
✅ GPU Name      : Tesla T4
✅ Compute Cap. : 7.5
✅ VRAM         : 14.56 GB

NVIDIA Driver
--------------------------------------------------------------------------------

✅ Environment Ready


In [6]:
# ==========================================================
# Cell 2 — Install Dependencies (Production V4)
# ==========================================================

import os
import sys
import subprocess
import importlib

print("=" * 80)
print("Installing Dependencies")
print("=" * 80)

# ----------------------------------------------------------
# Helper
# ----------------------------------------------------------

def run(cmd):
    print("\n>>>", " ".join(cmd))
    subprocess.run(cmd, check=True)

# ----------------------------------------------------------
# System Packages
# ----------------------------------------------------------

run(["apt-get", "update", "-qq"])

run([
    "apt-get",
    "install",
    "-y",
    "colmap",
    "ffmpeg",
    "libgl1",
    "libglib2.0-0"
])

# ----------------------------------------------------------
# Python Packages
# ----------------------------------------------------------

run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--upgrade",
    "pip",
    "setuptools",
    "wheel"
])

# ----------------------------------------------------------
# Remove old installs
# ----------------------------------------------------------

run([
    sys.executable,
    "-m",
    "pip",
    "uninstall",
    "-y",
    "nerfstudio",
    "gsplat"
])

# ----------------------------------------------------------
# Install Latest Compatible Packages
# ----------------------------------------------------------

run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--upgrade",
    "nerfstudio",
    "gsplat"
])

# ----------------------------------------------------------
# Verify Environment
# ----------------------------------------------------------

import torch

print("\n" + "=" * 80)
print("Environment")
print("=" * 80)

print("Python      :", sys.version.split()[0])
print("Torch       :", torch.__version__)
print("CUDA        :", torch.cuda.is_available())
print("GPU         :", torch.cuda.get_device_name(0))

try:
    ns = importlib.import_module("nerfstudio")
    print("Nerfstudio  :", getattr(ns, "__version__", "Installed"))
except Exception:
    print("Nerfstudio  : Installed")

try:
    gs = importlib.import_module("gsplat")
    print("gsplat      :", getattr(gs, "__version__", "Installed"))
except Exception:
    print("gsplat      : Installed")

print("\nChecking Commands...")

commands = [
    ["colmap", "-h"],
    ["ns-process-data", "--help"],
    ["ns-train", "--help"],
    ["ns-export", "--help"]
]

for cmd in commands:
    subprocess.run(
        cmd,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=True
    )

print("\n✅ All Dependencies Ready")

Installing Dependencies

>>> apt-get update -qq

>>> apt-get install -y colmap ffmpeg libgl1 libglib2.0-0

>>> /usr/bin/python3 -m pip install --upgrade pip setuptools wheel

>>> /usr/bin/python3 -m pip uninstall -y nerfstudio gsplat

>>> /usr/bin/python3 -m pip install --upgrade nerfstudio gsplat

Environment
Python      : 3.12.13
Torch       : 2.11.0+cu128
CUDA        : True
GPU         : Tesla T4
Nerfstudio  : Installed
gsplat      : 1.4.0

Checking Commands...

✅ All Dependencies Ready


In [7]:
# ==========================================================
# Cell 3 — Upload & Validate Images (Production V4)
# ==========================================================

import os
import shutil
import zipfile
from pathlib import Path
from PIL import Image
from google.colab import files

print("=" * 80)
print("Upload Image ZIP")
print("=" * 80)

# ----------------------------------------------------------
# Paths
# ----------------------------------------------------------

ZIP_NAME = "frames_for_colab.zip"

UPLOAD_DIR = Path("/content/upload")
EXTRACT_DIR = Path("/content/extracted")
IMAGES_DIR = Path("/content/images")

# ----------------------------------------------------------
# Cleanup
# ----------------------------------------------------------

for folder in [UPLOAD_DIR, EXTRACT_DIR, IMAGES_DIR]:
    shutil.rmtree(folder, ignore_errors=True)
    folder.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------
# Upload
# ----------------------------------------------------------

uploaded = files.upload()

if len(uploaded) == 0:
    raise RuntimeError("❌ No ZIP uploaded.")

uploaded_file = list(uploaded.keys())[0]

if not uploaded_file.lower().endswith(".zip"):
    raise RuntimeError("❌ Please upload a ZIP file.")

shutil.move(uploaded_file, UPLOAD_DIR / ZIP_NAME)

# ----------------------------------------------------------
# Extract
# ----------------------------------------------------------

print("\nExtracting ZIP...")

with zipfile.ZipFile(UPLOAD_DIR / ZIP_NAME, "r") as z:

    members = [
        m for m in z.namelist()
        if "__MACOSX" not in m
        and not Path(m).name.startswith(".")
    ]

    z.extractall(EXTRACT_DIR, members)

# ----------------------------------------------------------
# Collect Images
# ----------------------------------------------------------

extensions = {".jpg", ".jpeg", ".png"}

image_files = sorted([
    p for p in EXTRACT_DIR.rglob("*")
    if p.suffix.lower() in extensions
])

if len(image_files) == 0:
    raise RuntimeError("❌ No images found.")

# ----------------------------------------------------------
# Validate Images
# ----------------------------------------------------------

valid = []
invalid = []

sizes = []

for img in image_files:

    try:

        with Image.open(img) as im:
            im.verify()

        with Image.open(img) as im:
            sizes.append(im.size)

        valid.append(img)

    except Exception:

        invalid.append(img)

# ----------------------------------------------------------
# Copy + Rename
# ----------------------------------------------------------

for idx, img in enumerate(valid):

    ext = ".jpg"

    dst = IMAGES_DIR / f"{idx:05d}{ext}"

    shutil.copy2(img, dst)

# ----------------------------------------------------------
# Resolution Stats
# ----------------------------------------------------------

widths = [w for w, h in sizes]
heights = [h for w, h in sizes]

print("\n" + "=" * 80)
print("Dataset Summary")
print("=" * 80)

print(f"Valid Images     : {len(valid)}")
print(f"Invalid Images   : {len(invalid)}")

print(f"Min Width        : {min(widths)}")
print(f"Max Width        : {max(widths)}")

print(f"Min Height       : {min(heights)}")
print(f"Max Height       : {max(heights)}")

print("\nFirst 10 Images")

for img in sorted(IMAGES_DIR.iterdir())[:10]:
    print(img.name)

if len(valid) < 20:
    raise RuntimeError(
        "❌ Dataset too small. Need at least 20 valid images."
    )

print("\n✅ Images Ready")
print(IMAGES_DIR)

Upload Image ZIP


Saving frames_for_colab.zip to frames_for_colab (1).zip

Extracting ZIP...

Dataset Summary
Valid Images     : 125
Invalid Images   : 0
Min Width        : 1920
Max Width        : 1920
Min Height       : 1080
Max Height       : 1080

First 10 Images
00000.jpg
00001.jpg
00002.jpg
00003.jpg
00004.jpg
00005.jpg
00006.jpg
00007.jpg
00008.jpg
00009.jpg

✅ Images Ready
/content/images


In [10]:
# ==========================================================
# Cell 4 — COLMAP Reconstruction (Production V5)
# ==========================================================

import shutil
import subprocess
import torch
from pathlib import Path

print("=" * 80)
print("COLMAP Reconstruction")
print("=" * 80)

# ----------------------------------------------------------
# Paths
# ----------------------------------------------------------

IMAGE_DIR = Path("/content/images")
WORKSPACE = Path("/content/colmap")

DATABASE_PATH = WORKSPACE / "database.db"
SPARSE_PATH = WORKSPACE / "sparse"

# ----------------------------------------------------------
# Cleanup
# ----------------------------------------------------------

if WORKSPACE.exists():
    shutil.rmtree(WORKSPACE)

WORKSPACE.mkdir(parents=True, exist_ok=True)
SPARSE_PATH.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------
# Validate Dataset
# ----------------------------------------------------------

images = sorted(list(IMAGE_DIR.glob("*.jpg")))

if len(images) < 20:
    raise RuntimeError(
        f"Dataset too small ({len(images)} images). "
        "Need at least 20 images."
    )

print(f"Images Found : {len(images)}")

# ----------------------------------------------------------
# GPU Detection
# ----------------------------------------------------------

USE_GPU = "1" if torch.cuda.is_available() else "0"

print(f"GPU Available : {torch.cuda.is_available()}")
print(f"SIFT GPU      : {USE_GPU}")

# ----------------------------------------------------------
# Helper
# ----------------------------------------------------------

def run(cmd, step_name):
    print("\n" + "=" * 80)
    print(step_name)
    print("=" * 80)

    print(" ".join(map(str, cmd)))

    result = subprocess.run(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )

    print(result.stdout)

    if result.returncode != 0:
        raise RuntimeError(f"{step_name} failed.")

# ==========================================================
# STEP 1
# Feature Extraction
# ==========================================================

run(
    [
        "colmap",
        "feature_extractor",

        "--database_path", str(DATABASE_PATH),
        "--image_path", str(IMAGE_DIR),

        "--ImageReader.single_camera", "1",

        "--SiftExtraction.use_gpu", USE_GPU,

        "--SiftExtraction.max_num_features", "12000",

        "--SiftExtraction.estimate_affine_shape", "1",

        "--SiftExtraction.domain_size_pooling", "1"
    ],
    "STEP 1 - Feature Extraction"
)

if not DATABASE_PATH.exists():
    raise RuntimeError("database.db was not created.")

print("Database Created ✓")

# ==========================================================
# STEP 2
# Sequential Matcher
# ==========================================================

run(
    [
        "colmap",
        "sequential_matcher",

        "--database_path", str(DATABASE_PATH),

        "--SiftMatching.use_gpu", "0",

        "--SequentialMatching.overlap", "20",

        "--SiftMatching.guided_matching", "1",

        "--SiftMatching.num_threads", "2"
    ],
    "STEP 2 - Sequential Matching"
)

# ==========================================================
# STEP 3
# Mapper
# ==========================================================

run(
    [
        "colmap",
        "mapper",

        "--database_path", str(DATABASE_PATH),

        "--image_path", str(IMAGE_DIR),

        "--output_path", str(SPARSE_PATH),

        "--Mapper.multiple_models", "0",

        "--Mapper.init_min_num_inliers", "80",

        "--Mapper.abs_pose_min_num_inliers", "40",

        "--Mapper.filter_max_reproj_error", "4"
    ],
    "STEP 3 - Sparse Reconstruction"
)

# ----------------------------------------------------------
# Validate Reconstruction
# ----------------------------------------------------------

MODEL_PATH = SPARSE_PATH / "0"

if not MODEL_PATH.exists():
    raise RuntimeError(
        "Sparse model was not generated."
    )

required_files = [
    "cameras.bin",
    "images.bin",
    "points3D.bin"
]

missing = []

for f in required_files:
    if not (MODEL_PATH / f).exists():
        missing.append(f)

if missing:
    raise RuntimeError(
        f"Missing reconstruction files: {missing}"
    )

print("\n" + "=" * 80)
print("Reconstruction Summary")
print("=" * 80)

registered_images = len(list(images))
points_size = (MODEL_PATH / "points3D.bin").stat().st_size

print(f"Registered Images : {registered_images}")
print(f"points3D.bin Size : {points_size/1024:.2f} KB")

print("\nGenerated Files")

for f in sorted(MODEL_PATH.iterdir()):
    print("  •", f.name)

print("\nModel Location")
print(MODEL_PATH)

print("\n✅ COLMAP Reconstruction Completed Successfully")

Streaming output truncated to the last 5000 lines.
   6  1.078626e+03    2.36e-02    7.45e+02   1.08e+00   1.00e+00  7.29e+06        1    3.81e-03    2.72e-02
   7  1.078608e+03    1.73e-02    3.97e+02   2.99e-01   1.05e+00  2.19e+07        1    3.67e-03    3.09e-02
   8  1.078598e+03    1.01e-02    1.63e+03   7.15e-01   9.58e-01  6.56e+07        1    3.76e-03    3.47e-02
   9  1.078594e+03    4.51e-03    1.47e+03   7.56e-01   9.04e-01  1.39e+08        1    4.29e-03    3.90e-02
  10  1.078592e+03    1.57e-03    1.98e+02   2.75e-01   1.07e+00  4.17e+08        1    3.83e-03    4.29e-02
  11  1.078592e+03    9.51e-05    1.75e+01   6.85e-02   1.22e+00  1.25e+09        1    3.74e-03    4.66e-02
  12  1.078592e+03    8.34e-06    2.09e+00   1.30e-02   1.30e+00  3.75e+09        1    3.81e-03    5.05e-02
  13  1.078592e+03    8.30e-07    1.95e+00   2.94e-03   1.31e+00  1.13e+10        1    3.70e-03    5.42e-02
  14  1.078592e+03    8.23e-08    8.84e-01   8.40e-04   1.31e+00  3.38e+10        1  

In [11]:
# ==========================================================
# Cell 5 — COLMAP → Nerfstudio Dataset (Production V6)
# ==========================================================

import os
import shutil
import subprocess
from pathlib import Path
from importlib.metadata import version

print("=" * 80)
print("COLMAP → Nerfstudio Dataset Conversion")
print("=" * 80)

# ----------------------------------------------------------
# Paths
# ----------------------------------------------------------

IMAGE_DIR = Path("/content/images")
MODEL_DIR = Path("/content/colmap/sparse/0")
OUTPUT_DIR = Path("/content/ns_data")

# ----------------------------------------------------------
# Nerfstudio Version
# ----------------------------------------------------------

try:
    print(f"Nerfstudio Version : {version('nerfstudio')}")
except Exception:
    print("Nerfstudio Version : Unknown")

# ----------------------------------------------------------
# Validate Images
# ----------------------------------------------------------

if not IMAGE_DIR.exists():
    raise RuntimeError(f"Image directory not found:\n{IMAGE_DIR}")

images = sorted(
    list(IMAGE_DIR.glob("*.jpg")) +
    list(IMAGE_DIR.glob("*.jpeg")) +
    list(IMAGE_DIR.glob("*.png"))
)

if len(images) == 0:
    raise RuntimeError("No images found.")

print(f"Images Found : {len(images)}")

# ----------------------------------------------------------
# Validate COLMAP Model
# ----------------------------------------------------------

if not MODEL_DIR.exists():
    raise RuntimeError(f"COLMAP model not found:\n{MODEL_DIR}")

required = [
    "cameras.bin",
    "images.bin",
    "points3D.bin"
]

missing = []

for f in required:
    if not (MODEL_DIR / f).exists():
        missing.append(f)

if missing:
    raise RuntimeError(
        "COLMAP model incomplete.\nMissing:\n" +
        "\n".join(missing)
    )

print("COLMAP Model Verified ✓")

# ----------------------------------------------------------
# Clean Previous Output
# ----------------------------------------------------------

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

OUTPUT_DIR.mkdir(parents=True)

# ----------------------------------------------------------
# Build Command
# ----------------------------------------------------------

cmd = [
    "ns-process-data",
    "images",
    "--data", str(IMAGE_DIR),
    "--output-dir", str(OUTPUT_DIR),
    "--skip-colmap",
    "--colmap-model-path", str(MODEL_DIR),
]

print("\nRunning Command:\n")
print(" ".join(cmd))

# ----------------------------------------------------------
# Execute
# ----------------------------------------------------------

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for line in process.stdout:
    print(line, end="")

process.wait()

if process.returncode != 0:
    raise RuntimeError(
        f"\nns-process-data failed with exit code {process.returncode}"
    )

# ----------------------------------------------------------
# Validate Output
# ----------------------------------------------------------

TRANSFORMS = OUTPUT_DIR / "transforms.json"

if not TRANSFORMS.exists():

    print("\nCurrent Output Directory\n")

    for root, dirs, files in os.walk(OUTPUT_DIR):
        level = root.replace(str(OUTPUT_DIR), "").count(os.sep)
        indent = "    " * level

        print(f"{indent}{os.path.basename(root)}/")

        for f in files:
            print(f"{indent}    {f}")

    raise RuntimeError(
        "transforms.json was not generated."
    )

# ----------------------------------------------------------
# Dataset Summary
# ----------------------------------------------------------

print("\n" + "=" * 80)
print("Dataset Summary")
print("=" * 80)

print(f"Images        : {len(images)}")
print(f"Dataset Path  : {OUTPUT_DIR}")
print(f"Transforms    : {TRANSFORMS}")

print("\nGenerated Files\n")

for item in sorted(OUTPUT_DIR.iterdir()):
    print(" •", item.name)

print("\n✅ Nerfstudio dataset created successfully.")

COLMAP → Nerfstudio Dataset Conversion
Nerfstudio Version : 1.1.5
Images Found : 125
COLMAP Model Verified ✓

Running Command:

ns-process-data images --data /content/images --output-dir /content/ns_data --skip-colmap --colmap-model-path /content/colmap/sparse/0
[12:49:58] 🎉 Done copying images with prefix 'frame_'.                                        process_data_utils.py:348
[12:49:59] 🎉 🎉 🎉 All DONE 🎉 🎉 🎉                                                images_to_nerfstudio_dataset.py:135
           Starting with 125 images                                                  images_to_nerfstudio_dataset.py:138
           Colmap matched 49 images                                                  images_to_nerfstudio_dataset.py:138
           COLMAP only found poses for 39.20% of the images. This is low.            images_to_nerfstudio_dataset.py:138
           This can be caused by a variety of reasons, such poor scene coverage,                                        
           blurry 

In [13]:
# ==========================================================
# Cell 6 — Train Gaussian Splatting (Production V7)
# Compatible with Nerfstudio 1.1.5
# ==========================================================

import os
import shutil
import subprocess
from pathlib import Path
from importlib.metadata import version

# ----------------------------------------------------------
# Paths
# ----------------------------------------------------------

DATA_DIR = Path("/content/ns_data")
OUTPUT_DIR = Path("/content/splat_output")

print("=" * 80)
print("Gaussian Splatting Training")
print("=" * 80)

# ----------------------------------------------------------
# Nerfstudio Version
# ----------------------------------------------------------

try:
    print("Nerfstudio :", version("nerfstudio"))
except Exception:
    print("Nerfstudio : Unknown")

# ----------------------------------------------------------
# Verify Dataset
# ----------------------------------------------------------

if not DATA_DIR.exists():
    raise RuntimeError(f"Dataset not found:\n{DATA_DIR}")

TRANSFORMS = DATA_DIR / "transforms.json"

if not TRANSFORMS.exists():
    raise RuntimeError(
        "transforms.json not found.\nRun Cell 5 first."
    )

print("✅ Dataset Verified")
print("✅ transforms.json Found")

# ----------------------------------------------------------
# Clean Previous Output
# ----------------------------------------------------------

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

shutil.rmtree("/root/.cache/torch_extensions", ignore_errors=True)

# ----------------------------------------------------------
# Environment
# ----------------------------------------------------------

os.environ["MAX_JOBS"] = "1"
os.environ["CMAKE_BUILD_PARALLEL_LEVEL"] = "1"

print("\nEnvironment")
print("-" * 80)
print("MAX_JOBS =", os.environ["MAX_JOBS"])
print("CMAKE_BUILD_PARALLEL_LEVEL =", os.environ["CMAKE_BUILD_PARALLEL_LEVEL"])

# ----------------------------------------------------------
# GPU Info
# ----------------------------------------------------------

print("\nGPU")
print("-" * 80)
subprocess.run(["nvidia-smi"])

# ----------------------------------------------------------
# Training Command
# IMPORTANT:
# Training options BEFORE nerfstudio-data
# Dataset options AFTER nerfstudio-data
# ----------------------------------------------------------

cmd = [
    "ns-train",
    "splatfacto",

    "--output-dir",
    str(OUTPUT_DIR),

    "--max-num-iterations",
    "7000",

    "--pipeline.model.sh-degree",
    "0",

    "--viewer.quit-on-train-completion",
    "True",

    "--machine.num-devices",
    "1",

    "nerfstudio-data",

    "--data",
    str(DATA_DIR),
]

print("\n" + "=" * 80)
print("Training Command")
print("=" * 80)
print(" ".join(cmd))

# ----------------------------------------------------------
# Start Training
# ----------------------------------------------------------

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in process.stdout:
    print(line, end="")

process.wait()

if process.returncode != 0:
    raise RuntimeError(
        f"\n❌ Training failed.\nExit Code: {process.returncode}"
    )

# ----------------------------------------------------------
# Locate Outputs
# ----------------------------------------------------------

ckpts = sorted(OUTPUT_DIR.rglob("*.ckpt"))
configs = sorted(OUTPUT_DIR.rglob("config.yml"))
plys = sorted(OUTPUT_DIR.rglob("*.ply"))

print("\n" + "=" * 80)
print("Training Summary")
print("=" * 80)

print(f"Checkpoint Files : {len(ckpts)}")
print(f"Config Files     : {len(configs)}")
print(f"PLY Files        : {len(plys)}")

print("\nCheckpoint Files")
for f in ckpts:
    print(" •", f)

print("\nConfig Files")
for f in configs:
    print(" •", f)

print("\nPLY Files")
for f in plys:
    print(" •", f)

print("\n✅ Gaussian Splatting Training Completed Successfully.")

Streaming output truncated to the last 5000 lines.
3900 (55.71%)       14.938 ms            46 s, 308.936 ms     35.17 M                                
3910 (55.86%)       15.318 ms            47 s, 332.510 ms     34.34 M                                
3920 (56.00%)       14.799 ms            45 s, 581.944 ms     35.07 M                                
3930 (56.14%)       15.143 ms            46 s, 488.940 ms     34.28 M                                
---------------------------------------------------------------------------------------------------- 
Viewer running locally at: http://localhost:7007 (listening on 0.0.0.0)                              


Step (% Done)       Train Iter (time)    ETA (time)           Train Rays / Sec                       
-----------------------------------------------------------------------------------                  
3850 (55.00%)       15.030 ms            47 s, 344.133 ms     34.59 M                                
3860 (55.14%)       14.619 ms

In [15]:
# ==========================================================
# Cell 7A — Patch Nerfstudio (PyTorch 2.6 Compatibility)
# ==========================================================

import os
import re
from pathlib import Path

patch_file = Path("/usr/local/lib/python3.12/dist-packages/nerfstudio/utils/eval_utils.py")

if not patch_file.exists():
    raise FileNotFoundError(f"Cannot find {patch_file}")

print("Patching:")
print(patch_file)

text = patch_file.read_text()

old = "loaded_state = torch.load(load_path, map_location=\"cpu\")"
new = "loaded_state = torch.load(load_path, map_location=\"cpu\", weights_only=False)"

if new in text:
    print("✅ Patch already applied.")

elif old in text:
    text = text.replace(old, new)
    patch_file.write_text(text)
    print("✅ Patch applied successfully.")

else:
    print("❌ Expected line not found.")
    print("Searching for torch.load...")

    for i, line in enumerate(text.splitlines(), start=1):
        if "torch.load(" in line:
            print(f"{i}: {line}")

    raise RuntimeError("Automatic patch failed.")

print("\nDone.")

Patching:
/usr/local/lib/python3.12/dist-packages/nerfstudio/utils/eval_utils.py
✅ Patch applied successfully.

Done.


In [16]:
# ==========================================================
# Cell 7 — Export Gaussian Splat (Production V8)
# Compatible with Nerfstudio 1.1.x
# ==========================================================

import os
import glob
import shutil
import subprocess
from pathlib import Path

print("=" * 80)
print("Gaussian Splat Export")
print("=" * 80)

# ----------------------------------------------------------
# Paths
# ----------------------------------------------------------

OUTPUT_ROOT = Path("/content/splat_output")
EXPORT_DIR = Path("/content/splat_export")

# ----------------------------------------------------------
# Clean Previous Export
# ----------------------------------------------------------

if EXPORT_DIR.exists():
    shutil.rmtree(EXPORT_DIR)

EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------
# Find Latest Nerfstudio Config
# ----------------------------------------------------------

configs = sorted(
    glob.glob(
        "/content/splat_output/**/config.yml",
        recursive=True
    ),
    key=os.path.getmtime
)

if len(configs) == 0:
    raise FileNotFoundError(
        "\n❌ No config.yml found.\n"
        "Did Cell 6 finish successfully?"
    )

config_path = configs[-1]

print("\nLatest Config")
print("-" * 80)
print(config_path)

# ----------------------------------------------------------
# Export Command
# ----------------------------------------------------------

cmd = [
    "ns-export",
    "gaussian-splat",
    "--load-config",
    config_path,
    "--output-dir",
    str(EXPORT_DIR),
]

print("\n" + "=" * 80)
print("Running Export")
print("=" * 80)
print(" ".join(cmd))
print()

# ----------------------------------------------------------
# Run Export (Live Logs)
# ----------------------------------------------------------

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in process.stdout:
    print(line, end="")

process.wait()

if process.returncode != 0:
    raise RuntimeError(
        f"\n❌ Gaussian Export Failed\n"
        f"Exit Code : {process.returncode}"
    )

# ----------------------------------------------------------
# Verify Export
# ----------------------------------------------------------

print("\n" + "=" * 80)
print("Export Summary")
print("=" * 80)

all_files = sorted(EXPORT_DIR.rglob("*"))

if len(all_files) == 0:
    raise RuntimeError(
        "❌ Export directory is empty."
    )

print(f"\nGenerated Files : {len(all_files)}\n")

for f in all_files:
    if f.is_file():
        print("•", f)

# ----------------------------------------------------------
# Search PLY
# ----------------------------------------------------------

ply_files = sorted(EXPORT_DIR.rglob("*.ply"))

print("\n" + "-" * 80)

if len(ply_files) > 0:
    print("✅ Gaussian Export Successful")
    print("\nPLY Files:\n")

    for ply in ply_files:
        print(ply)

else:
    print("⚠️ No .ply file found.")
    print("Some Nerfstudio versions export")
    print("other Gaussian formats instead.")

print("\nExport Folder")
print("-" * 80)
print(EXPORT_DIR)

print("\n🎉 Cell 7 Completed Successfully.")

Gaussian Splat Export

Latest Config
--------------------------------------------------------------------------------
/content/splat_output/unnamed/splatfacto/2026-07-28_125554/config.yml

Running Export
ns-export gaussian-splat --load-config /content/splat_output/unnamed/splatfacto/2026-07-28_125554/config.yml --output-dir /content/splat_export

2026-07-28 13:40:53.722662: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.12/dist-packages/nerfstudio/field_components/activations.py:32: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd(cast_inputs=torch.float32)
/usr/local/lib/python3.12/dist-packages/nerfstudio/field_compone

In [17]:
# ==========================================================
# Cell 8 — Package & Download Results (Production V2)
# ==========================================================

import os
import zipfile
import hashlib
from pathlib import Path
from google.colab import files

print("=" * 80)
print("Package Gaussian Splat Export")
print("=" * 80)

EXPORT_DIR = Path("/content/splat_export")
ZIP_FILE = Path("/content/gaussian_splat_export.zip")

# ----------------------------------------------------------
# Verify Export Folder
# ----------------------------------------------------------

if not EXPORT_DIR.exists():
    raise RuntimeError(
        f"❌ Export folder not found:\n{EXPORT_DIR}\nRun Cell 7 first."
    )

export_files = sorted(
    [f for f in EXPORT_DIR.rglob("*") if f.is_file()]
)

if len(export_files) == 0:
    raise RuntimeError("❌ Export folder is empty.")

# ----------------------------------------------------------
# Remove Old ZIP
# ----------------------------------------------------------

if ZIP_FILE.exists():
    ZIP_FILE.unlink()

# ----------------------------------------------------------
# Create ZIP
# ----------------------------------------------------------

print("\nCreating ZIP...\n")

with zipfile.ZipFile(
    ZIP_FILE,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zipf:

    for file in export_files:
        zipf.write(
            file,
            arcname=file.relative_to(EXPORT_DIR)
        )

# ----------------------------------------------------------
# Verify ZIP
# ----------------------------------------------------------

if not ZIP_FILE.exists():
    raise RuntimeError("❌ ZIP creation failed.")

zip_size = ZIP_FILE.stat().st_size / (1024 * 1024)

if zip_size == 0:
    raise RuntimeError("❌ ZIP file is empty.")

# ----------------------------------------------------------
# SHA256
# ----------------------------------------------------------

sha256 = hashlib.sha256()

with open(ZIP_FILE, "rb") as f:
    while True:
        chunk = f.read(1024 * 1024)
        if not chunk:
            break
        sha256.update(chunk)

checksum = sha256.hexdigest()

# ----------------------------------------------------------
# Summary
# ----------------------------------------------------------

print("=" * 80)
print("Package Summary")
print("=" * 80)

print(f"Export Folder : {EXPORT_DIR}")
print(f"ZIP File      : {ZIP_FILE}")
print(f"Files         : {len(export_files)}")
print(f"ZIP Size      : {zip_size:.2f} MB")

print("\nContents:\n")

for file in export_files:
    print("•", file.relative_to(EXPORT_DIR))

print("\nSHA256:")
print(checksum)

print("\nDestination after extraction:")
print("data/scans/<scan_id>/splat/")

print("\nDownloading ZIP...\n")

# ----------------------------------------------------------
# Download
# ----------------------------------------------------------

files.download(str(ZIP_FILE))

print("=" * 80)
print("🎉 Gaussian Splatting Pipeline Completed Successfully!")
print("=" * 80)

Package Gaussian Splat Export

Creating ZIP...

Package Summary
Export Folder : /content/splat_export
ZIP File      : /content/gaussian_splat_export.zip
Files         : 1
ZIP Size      : 8.33 MB

Contents:

• splat.ply

SHA256:
73f5f0c3f81b6ff3ca541b2cdf6023179f33f896ef13972da395ecf47607a4ac

Destination after extraction:
data/scans/<scan_id>/splat/




<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

🎉 Gaussian Splatting Pipeline Completed Successfully!
